# COLETA DESPESAS


In [ ]:
import pandas as pd

def gerar_arquivao_despesas(anos):
    frames = []

    for ano in anos:
        print(f"📥 Baixando todas as despesas de {ano}...")

        url = f"http://www.camara.leg.br/cotas/Ano-{ano}.csv.zip"

        try:

            df = pd.read_csv(url, sep=';', low_memory=False, compression='zip', encoding='utf-8')


            df = df.dropna(subset=['ideCadastro'])


            df['vlrLiquido'] = pd.to_numeric(df['vlrLiquido'], errors='coerce').fillna(0)
            df = df[df['vlrLiquido'] > 0]


            frames.append(df[['ideCadastro', 'txtDescricao', 'vlrLiquido']])

        except UnicodeDecodeError:

            df = pd.read_csv(url, sep=';', low_memory=False, compression='zip', encoding='latin1')
            df = df.dropna(subset=['ideCadastro'])
            df['vlrLiquido'] = pd.to_numeric(df['vlrLiquido'], errors='coerce').fillna(0)
            df = df[df['vlrLiquido'] > 0]
            frames.append(df[['ideCadastro', 'txtDescricao', 'vlrLiquido']])
        except Exception as e:
            print(f" Erro no ano {ano}: {e}")

    if not frames:
        return pd.DataFrame()

    print("\n Processando e somando os gastos históricos...")
    df_tudo = pd.concat(frames, ignore_index=True)

    gastos_totais = df_tudo.groupby('ideCadastro')['vlrLiquido'].sum().reset_index()
    gastos_totais = gastos_totais.rename(columns={'vlrLiquido': 'Total_Gasto_Historico'})
    gastos_totais['Total_Gasto_Historico'] = gastos_totais['Total_Gasto_Historico'].round(2)

    print(" Mapeando as categorias e montando o dicionário de despesas...")
    gastos_cat = df_tudo.groupby(['ideCadastro', 'txtDescricao'])['vlrLiquido'].sum().reset_index()


    categorias_agrupadas = gastos_cat.groupby('ideCadastro').apply(
        lambda x: str(dict(zip(x['txtDescricao'], x['vlrLiquido'].round(2))))
    ).reset_index(name='Categoria_Despesa_Historico')


    df_final = pd.merge(gastos_totais, categorias_agrupadas, on='ideCadastro', how='inner')


    df_final = df_final.rename(columns={'ideCadastro': 'deputado_id'})
    df_final['deputado_id'] = df_final['deputado_id'].astype(int)

    return df_final


anos_historicos = list(range(2020, 2027))
df_modulo3 = gerar_arquivao_despesas(anos_historicos)

df_modulo3.to_csv("modulo3_despesas_completa.csv", index=False, encoding='utf-8')
print("\n Módulo 3 Finalizado! Arquivo de despesas históricas gerado.")